# Loading DGCI embeddings:

In [ ]:
%matplotlib widget    
import numpy as np                                          # already imported in cell 1
from sklearn.cluster import KMeans                          # the clustering used in the paper
from sklearn.decomposition import PCA                       # only for the 2D plot
from sklearn.metrics import silhouette_score                # cluster quality without labels
from sklearn.metrics import adjusted_rand_score             # agreement between two clusterings
import matplotlib.pyplot as plt   
import pandas as pd                                              # tables keyed by subject
import morphology_projection as mp  

codes_path = "/home/ids/gmargari-24/airway_project/DGCI/logs/unregistered_128_2M_4k/latent_codes.npy"
enc_output = np.load(codes_path).astype(np.float64)         # (419, 128); float64 keeps ward's sums stable

In [ ]:
codes      = enc_output                                     # (419, 128)
dim_median = np.median(codes, axis=0)                       # typical value of each dimension
dim_mad    = np.median(np.abs(codes - dim_median), axis=0)  # robust spread, ignores outlier shapes
mad_order  = np.argsort(dim_mad)[::-1]                      # most to least spread ([::-1] = reverse)

plt.figure(figsize=(8, 4))                                  # one wide panel
plt.semilogy(dim_mad[mad_order], marker=".")                # log scale exposes the gap
plt.xlabel("dimension rank")                                # sorted, not original index
plt.ylabel("MAD across shapes (log)")                       # spread per dimension
plt.show()                                                  # render

In [ ]:
pca_full      = PCA().fit(codes)                              # all 128 principal directions
eigvals       = pca_full.explained_variance_                  # variance along each direction
participation = eigvals.sum() ** 2 / (eigvals ** 2).sum()     # effective number of dimensions
cum_share     = np.cumsum(pca_full.explained_variance_ratio_) # cumulative variance explained
n_95          = np.searchsorted(cum_share, 0.95) + 1          # directions needed for 95% variance
print(f"participation ratio {participation:.1f}, 95% variance in {n_95} directions")  # summary

plt.figure(figsize=(8, 4))                                  # one wide panel
plt.semilogy(eigvals, marker=".")                           # look for where the spectrum drops
plt.xlabel("principal component")                           # ordered by variance
plt.ylabel("variance (log)")                                # per-direction variance
plt.show()                                                  # render

In [ ]:
active_dims     = np.where(dim_mad > 3e-4)[0]                        # threshold sits in the gap
borderline_dims = np.where((dim_mad > 1e-4) & (dim_mad <= 3e-4))[0]  # ranks ~46–49
print(len(active_dims), active_dims)                                 # expect roughly 45
print(len(borderline_dims), borderline_dims)                         # confirm these with ablation

In [ ]:
from sklearn.cluster import KMeans                          # clustering
from sklearn.decomposition import PCA                       # 2D plot only
from sklearn.metrics import silhouette_score                # cluster quality without labels
from sklearn.metrics import adjusted_rand_score             # agreement between two clusterings
import matplotlib.pyplot as plt                             # plotting

dim_median   = np.median(enc_output, axis=0)                 # typical value per dimension
dim_mad      = np.median(np.abs(enc_output - dim_median), axis=0)  # robust spread per dimension
active_dims  = np.where(dim_mad > 3e-4)[0]                   # threshold sits in the gap of your plot
wide_dims    = np.where(dim_mad > 1e-4)[0]                   # also includes the borderline ones

codes_active = enc_output[:, active_dims]                   # (419, ~45)
print(codes_active.shape, len(wide_dims))                   # confirm the counts

In [ ]:
k_values    = list(range(2, 11))                            # candidate numbers of clusters
inertias    = []                                            # within-cluster spread per K
silhouettes = []                                            # separation score per K

for k in k_values:                                          # one fit per candidate K
    km  = KMeans(n_clusters=k, n_init=20, random_state=0)   # n_init = restarts, best one kept
    ids = km.fit_predict(codes_active)                      # cluster the active dimensions only
    inertias.append(km.inertia_)                            # for the elbow plot
    silhouettes.append(silhouette_score(codes_active, ids)) # higher = better separated

fig, axes = plt.subplots(1, 2, figsize=(10, 4))             # elbow and silhouette side by side
axes[0].plot(k_values, inertias, marker="o")                # look for the bend
axes[0].set_xlabel("K")                                     # number of clusters
axes[0].set_ylabel("Inertia")                               # lower = tighter
axes[1].plot(k_values, silhouettes, marker="o")             # look for the peak
axes[1].set_xlabel("K")                                     # number of clusters
axes[1].set_ylabel("Silhouette")                            # higher = better separated
plt.show()                                                  # render

In [ ]:
n_clusters = 4                                        # replace with the K chosen above

kmeans      = KMeans(n_clusters=n_clusters, n_init=20, random_state=0) # 
cluster_ids = kmeans.fit_predict(codes_active)              # cluster id for each of the 419 shapes
print(np.bincount(cluster_ids))                             # shapes per cluster (bincount = counts)

pca        = PCA(n_components=2)                            # plot axes from the active dims
codes_2d   = pca.fit_transform(codes_active)                # (419, 2)
centers_2d = pca.transform(kmeans.cluster_centers_)         # centroids on the same axes
print(pca.explained_variance_ratio_)                        # how much the 2D view shows

plt.figure(figsize=(6, 5))                                  # single panel like Fig. 3
plt.scatter(codes_2d[:, 0], codes_2d[:, 1], c=cluster_ids, cmap="tab10", alpha=0.6)  # color = cluster
plt.scatter(centers_2d[:, 0], centers_2d[:, 1], c="black", marker="x", s=100)  # centroids
plt.xlabel("PCA1")                                          # first component
plt.ylabel("PCA2")                                          # second component
plt.show()                                                  # render

In [ ]:
def plot_clusters(xy, title, centers=None):
    """Scatter 2D points colored by K-means cluster, with a legend."""
    plt.figure(figsize=(7, 6))                              # slightly larger canvas
    for c in range(n_clusters):                             # one call per cluster gives a legend
        pts = xy[cluster_ids == c]                          # points in cluster c
        # smaller markers without edges overlap less
        plt.scatter(pts[:, 0], pts[:, 1], s=18, alpha=0.7, edgecolors="none", label=f"cluster {c}")
    if centers is not None:                                 # t-SNE has no centroids to draw
        plt.scatter(centers[:, 0], centers[:, 1], c="black", marker="x", s=120)  # centroids
    plt.title(title)                                        # which projection this is
    plt.legend()                                            # cluster names

In [ ]:
low = np.percentile(codes_2d, 1, axis=0)                    # 1st percentile on each axis
high = np.percentile(codes_2d, 99, axis=0)                  # 99th percentile on each axis
margin = 0.1 * (high - low)                                 # a little padding around the box

plot_clusters(codes_2d, "PCA (zoomed)", centers_2d)         # same data as before
plt.xlim(low[0] - margin[0], high[0] + margin[0])           # crop the far outliers horizontally
plt.ylim(low[1] - margin[1], high[1] + margin[1])           # and vertically
plt.xlabel("PCA1")                                          # first component
plt.ylabel("PCA2")                                          # second component
plt.show()                                                  # render

### Active latent codes
       │
       ├──────────────► K-means ───► cluster labels
       │
       └──────────────► Kernel PCA ─► 3D visualization

In [ ]:
# from sklearn.discriminant_analysis import LinearDiscriminantAnalysis  # supervised projection

# # shrinkage="auto" keeps the covariance estimate stable with ~45 dims and 419 shapes
# lda         = LinearDiscriminantAnalysis(n_components=2, solver="eigen", shrinkage="auto")
# codes_lda   = lda.fit_transform(codes_active, cluster_ids)  # at most K-1 = 3 components available
# centers_lda = lda.transform(kmeans.cluster_centers_)        # centroids on the same axes

# plot_clusters(codes_lda, "LDA on K-means clusters", centers_lda)  # separation-maximizing view
# plt.xlabel("LD1")                                           # best separating direction
# plt.ylabel("LD2")                                           # second best
# plt.show()                                                  # render

In [ ]:
import pandas as pd                                              # tables keyed by subject
import morphology_projection as mp                               # helper module, same folder as the notebook

log_dir         = "/home/ids/gmargari-24/airway_project/DGCI/logs/binary_128_2M_4k"  # DGCI run folder
latent_csv      = f"{log_dir}/latent_codes.csv"                  # subject ids in latent_codes.npy row order
metrics_csv     = f"{log_dir}/morphology/mean_morphological_features.csv"  # saved by clustering.ipynb
tree_counts_csv = "/home/ids/gmargari-24/airway_project/airmorph_cases/tree_counts.csv"  # branch/generation counts
kmeans_dir      = f"{log_dir}/morphology_kmeans_k{n_clusters}"   # own folder, so ward outputs aren't overwritten

subjects = pd.read_csv(latent_csv)["subject"].tolist()           # row i here == row i of enc_output
assert len(subjects) == len(cluster_ids), "labels and subjects differ in length"  # catch a wrong file early

metrics = pd.read_csv(metrics_csv)                               # subject + the 6 morphology means
assert metrics["subject"].tolist() == subjects, "metrics not in latent row order"  # alignment guard

counts = mp.build_tree_count_table(subjects, tree_counts_csv)    # subject + n_branches + max_generation

# 8 means per K-means cluster, plus a Kruskal-Wallis test per metric
wide, detailed, merged = mp.cluster_summary(cluster_ids, metrics, counts, save_dir=kmeans_dir)

print("\nmean of every metric per K-means cluster:")                # header for the compact table
print(wide.to_string())                                          # n + the 8 means, one row per cluster

print("\nwith spread and valid counts:")                         # header for the detailed table
print(detailed.to_string())                                      # mean / sd / valid n per metric

heatmap_path = mp.plot_cluster_heatmap(merged, save_dir=kmeans_dir)  # same means on a common SD scale
print(heatmap_path)                                              # where the PNG was written

                  DGCI latent codes
                         │
                         ▼
               Active latent dimensions
                         │
               ┌─────────┴─────────┐
               │                   │
               ▼                   ▼
            K-means             Other clustering
               │
               ▼
          Cluster labels
               │
       ┌───────┼────────┐
       │       │        │
       ▼       ▼        ▼
      PCA    KPCA     UMAP
      3D      3D       3D

In [ ]:
def centroid_projection(data, centers):
    """Coordinates of data and centroids in the 3D space through the centroids."""
    center_mean = centers.mean(axis=0)                      # middle point of the 4 centroids
    offsets = centers - center_mean                         # each centroid relative to that middle
    _, _, vt = np.linalg.svd(offsets, full_matrices=False)  # directions the offsets span (rows of vt)
    basis = vt[:3]                                          # 4 centroids span at most 3 directions
    data_3d = (data - center_mean) @ basis.T                # points in 3D (@ = matrix product)
    centers_3d = offsets @ basis.T                          # centroids on the same axes
    return data_3d, centers_3d                              # both share one coordinate system


def plot_clusters_3d(xyz, labels, title, centers=None):
    """3D scatter colored by cluster, cropped to the 1st-99th percentile."""
    fig = plt.figure(figsize=(8, 7))                        # one large panel
    ax = fig.add_subplot(projection="3d")                   # 3D axes
    for c in range(n_clusters):                             # one call per cluster gives a legend
        pts = xyz[labels == c]                              # points in cluster c
        ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=14, alpha=0.7, label=f"cluster {c}")
    if centers is not None:                                 # draw centroids when available
        ax.scatter(centers[:, 0], centers[:, 1], centers[:, 2], c="black", marker="x", s=150)
    low = np.percentile(xyz, 1, axis=0)                     # crop far outliers, as in your 2D zoom
    high = np.percentile(xyz, 99, axis=0)                   # upper bound per axis
    ax.set_xlim(low[0], high[0])                            # axis 1 range
    ax.set_ylim(low[1], high[1])                            # axis 2 range
    ax.set_zlim(low[2], high[2])                            # axis 3 range
    ax.set_title(title)                                     # which view this is
    ax.legend()                                             # cluster names
    return ax                                               # lets the caller change the angle

In [ ]:
codes_3d, centers_3d = centroid_projection(codes_active, kmeans.cluster_centers_)  # real data

# sanity check: nearest centroid in 3D should equal the 46-D label for every shape
dists_3d = np.linalg.norm(codes_3d[:, None, :] - centers_3d[None, :, :], axis=2)  # (419, 4) distances
print("same assignment in 3D:", (dists_3d.argmin(axis=1) == cluster_ids).mean())  # expect 1.0

ax = plot_clusters_3d(codes_3d, cluster_ids, "K-means clusters, centroid subspace", centers_3d)
ax.view_init(elev=20, azim=40)                              # camera angle (elevation, rotation), try others
plt.show()                                                  # render

In [ ]:
# ============================================================
# 3D PCA / 3D Kernel PCA / 3D UMAP
# Visualization of the SAME K-means cluster labels
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA, KernelPCA
from sklearn.preprocessing import StandardScaler


# ------------------------------------------------------------
# 1. 3D PCA
# ------------------------------------------------------------

pca_3d = PCA(n_components=3)

codes_pca_3d = pca_3d.fit_transform(codes_active)
centers_pca_3d = pca_3d.transform(kmeans.cluster_centers_)

print("3D PCA explained variance ratio:")
print(pca_3d.explained_variance_ratio_)
print("Total:", pca_3d.explained_variance_ratio_.sum())


fig = plt.figure(figsize=(9, 8))
ax = fig.add_subplot(projection="3d")

for c in range(n_clusters):
    pts = codes_pca_3d[cluster_ids == c]

    ax.scatter(
        pts[:, 0],
        pts[:, 1],
        pts[:, 2],
        s=18,
        alpha=0.7,
        label=f"cluster {c}"
    )

ax.scatter(
    centers_pca_3d[:, 0],
    centers_pca_3d[:, 1],
    centers_pca_3d[:, 2],
    c="black",
    marker="x",
    s=180,
    linewidths=3
)

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
ax.set_title("K-means clusters — 3D PCA")
ax.legend()

ax.view_init(elev=20, azim=40)

plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# 2. 3D Kernel PCA
# ------------------------------------------------------------

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import KernelPCA

# Standardize the active latent representation
scaler = StandardScaler()

codes_active_scaled = scaler.fit_transform(codes_active)


# RBF Kernel PCA
kpca_3d = KernelPCA(
    n_components=3,
    kernel="rbf",
    gamma=0.03,
    random_state=42
)

codes_kpca_3d = kpca_3d.fit_transform(codes_active_scaled)


fig = plt.figure(figsize=(9, 8))
ax = fig.add_subplot(projection="3d")

for c in range(n_clusters):
    pts = codes_kpca_3d[cluster_ids == c]

    ax.scatter(
        pts[:, 0],
        pts[:, 1],
        pts[:, 2],
        s=18,
        alpha=0.7,
        label=f"cluster {c}"
    )

ax.set_xlabel("Kernel PC1")
ax.set_ylabel("Kernel PC2")
ax.set_zlabel("Kernel PC3")
ax.set_title("K-means clusters — 3D Kernel PCA")
ax.legend()

ax.view_init(elev=20, azim=40)

plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# 3. 3D UMAP
# ------------------------------------------------------------

import umap.umap_ as umap

umap_3d = umap.UMAP(
    n_components=3,
    n_neighbors=30,
    min_dist=0.1,
    metric="euclidean",
    random_state=42
)

codes_umap_3d = umap_3d.fit_transform(codes_active)


fig = plt.figure(figsize=(9, 8))
ax = fig.add_subplot(projection="3d")

for c in range(n_clusters):
    pts = codes_umap_3d[cluster_ids == c]

    ax.scatter(
        pts[:, 0],
        pts[:, 1],
        pts[:, 2],
        s=18,
        alpha=0.7,
        label=f"cluster {c}"
    )

ax.set_xlabel("UMAP1")
ax.set_ylabel("UMAP2")
ax.set_zlabel("UMAP3")
ax.set_title("K-means clusters — 3D UMAP")
ax.legend()

ax.view_init(elev=20, azim=40)

plt.tight_layout()
plt.show()